In [ ]:
!pip install streamlit pyngrok

In [ ]:
# Create Base App File

%%writefile app.py
import streamlit as st
import random

st.title("🪨📄✂️ Rock Paper Scissors Game")


In [ ]:
# Add Images for Choices

# Images
%%writefile -a app.py
stone_img = "https://img.magnific.com/free-psd/grey-boulder-rock-isolated-transparent-background_632498-25568.jpg?semt=ais_hybrid&w=740&q=80"
paper_img = "https://static.vecteezy.com/system/resources/thumbnails/008/952/117/small/old-parchment-paper-sheet-vintage-aged-or-texture-isolated-on-white-background-photo.jpg"
scissor_img = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQ2RIUvkqFxR80_Cs39NbjHT1iob_xIEbeCMqKXrJvN3A&s=10"
choices = {"Stone": stone_img, "Paper": paper_img, "Scissor": scissor_img}

In [ ]:
# Initialize Session State

# Session state
%%writefile -a app.py
if "user_score" not in st.session_state:
    st.session_state.user_score = 0
    st.session_state.comp_score = 0
    st.session_state.draws = 0
    st.session_state.leaderboard = []
    st.session_state.rounds_played = 0
    st.session_state.best_of = 5


In [ ]:
# Sidebar Settings

%%writefile -a app.py
st.sidebar.header("⚙️ Game Settings")
st.session_state.best_of = st.sidebar.selectbox("Select Best of N rounds:", [3, 5, 7])

In [ ]:
# Always define user_choice so it's safe to reference later

%%writefile -a app.py
user_choice = None

if st.session_state.rounds_played >= st.session_state.best_of:
    st.subheader("🏁 Final Result")
    if st.session_state.user_score > st.session_state.comp_score:
        st.success("👑 You are the Champion!")
    elif st.session_state.comp_score > st.session_state.user_score:
        st.error("🤖 Computer Wins the Match!")
    else:
        st.warning("It's a Tie Overall!")

    if st.button("🏆 Save to Leaderboard"):
        st.session_state.leaderboard.append({
            "You": st.session_state.user_score,
            "Computer": st.session_state.comp_score,
            "Draws": st.session_state.draws
        })
        st.session_state.user_score = st.session_state.comp_score = st.session_state.draws = 0
        st.session_state.rounds_played = 0

    st.info("✅ Best of N rounds completed! No more moves allowed.")

else:
    col1, col2, col3 = st.columns(3)
    if col1.button("🪨 Stone"):
        user_choice = "Stone"
    if col2.button("📄 Paper"):
        user_choice = "Paper"
    if col3.button("✂️ Scissor"):
        user_choice = "Scissor"


In [ ]:
# Game Logic

%%writefile -a app.py
if user_choice:
    comp_choice = random.choice(list(choices.keys()))
    col1, col2 = st.columns(2)
    with col1:
        st.image(choices[user_choice], caption="You 🧑", width=250)
    with col2:
        st.image(choices[comp_choice], caption="Computer 🤖", width=250)

    if user_choice == comp_choice:
        st.warning("It's a Draw!")
        st.session_state.draws += 1
    elif (user_choice == "Stone" and comp_choice == "Scissor") or \
         (user_choice == "Paper" and comp_choice == "Stone") or \
         (user_choice == "Scissor" and comp_choice == "Paper"):
        st.success("🎉 You Win!")
        st.session_state.user_score += 1
    else:
        st.error("🤖 Computer Wins!")
        st.session_state.comp_score += 1

    st.session_state.rounds_played += 1


In [ ]:
# Scorecard

%%writefile -a app.py
st.subheader("📊 Scoreboard")
st.write(f"You: {st.session_state.user_score} | Computer: {st.session_state.comp_score} | Draws: {st.session_state.draws}")
st.write(f"Rounds Played: {st.session_state.rounds_played} / Best of {st.session_state.best_of}")

st.bar_chart({
    "You": [st.session_state.user_score],
    "Computer": [st.session_state.comp_score],
    "Draws": [st.session_state.draws]
})


In [ ]:
# Leaderboard + Replay

%%writefile -a app.py
st.subheader("🏆 Leaderboard")
for i, entry in enumerate(st.session_state.leaderboard, 1):
    st.write(f"{i}. You: {entry['You']} | Computer: {entry['Computer']} | Draws: {entry['Draws']}")

if st.button("🔄 Play Again"):
    st.session_state.user_score = 0
    st.session_state.comp_score = 0
    st.session_state.draws = 0
    st.session_state.rounds_played = 0


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3H8eEc2OqnXwfZznN9GwfgFFsgg_4dkWvZkR3LAQEYTcZRBX3")
import subprocess

# Start Streamlit in background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Open ngrok tunnel
# Replace "YOUR_NOGROK_AUTH_TOKEN" with a valid authtoken from your ngrok dashboard

public_url = ngrok.connect(8501)
print("Streamlit app is live at:", public_url)